In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import GradientBoostingClassifier

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Ucitavanje podataka

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

# uklanjanje ID
train = train.drop("id", axis=1)
test = test.drop("id", axis=1)

# uklanjanje NaN
train = train.dropna()
test = test.dropna()

# X i y
X_train = train.drop("satisfaction", axis=1)
y_train = train["satisfaction"]

X_test = test.drop("satisfaction", axis=1)
y_test = test["satisfaction"]

# Preprocessing

categorical_cols = ["Gender", "Customer Type", "Type of Travel", "Class"]
numerical_cols = [col for col in X_train.columns if col not in categorical_cols]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numerical_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
])

# Modeli i parametri

models = {
    "KNN": {
        "model": KNeighborsClassifier(),
        "params": {
            "model__n_neighbors": [3,5,7],
            "model__weights": ["uniform", "distance"]
        }
    },

    "SVM": {
        "model": SVC(),
        "params": {
            "model__C": [0.1, 1, 10],
            "model__kernel": ["linear", "rbf"]
        }
    },

    "MLP": {
        "model": MLPClassifier(max_iter=300),
        "params": {
            "model__hidden_layer_sizes": [(50,), (100,)],
            "model__activation": ["relu", "tanh"]
        }
    },

    "GradientBoosting": {
        "model": GradientBoostingClassifier(),
        "params": {
            "model__n_estimators": [100, 200],
            "model__learning_rate": [0.05, 0.1]
        }
    }
}

results = {}

# Treniranje i evaluacija

for name, config in models.items():
    print(f"\n===== {name} =====")

    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", config["model"])
    ])

    grid = GridSearchCV(
        pipe,
        config["params"],
        cv=5,
        scoring="accuracy",
        n_jobs=-1
    )

    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_

    y_pred = best_model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)

    print("Best params:", grid.best_params_)
    print("Accuracy:", acc)
    print(classification_report(y_test, y_pred))

    results[name] = acc

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)

    plt.figure()
    sns.heatmap(cm, annot=True, fmt='d')
    plt.title(f"Confusion Matrix - {name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

    # Feature Importance (za boosting)
    if name == "GradientBoosting":
        model = best_model.named_steps["model"]
        importances = model.feature_importances_

        feature_names = best_model.named_steps["preprocessor"].get_feature_names_out()

        indices = np.argsort(importances)[-10:]

        plt.figure()
        plt.barh(range(len(indices)), importances[indices])
        plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
        plt.title("Feature Importance - Gradient Boosting")
        plt.xlabel("Importance")
        plt.ylabel("Features")
        plt.show()

# Poredjenje modela

plt.figure()
plt.bar(results.keys(), results.values())
plt.title("Comparison of Models (Accuracy)")
plt.xlabel("Model")
plt.ylabel("Accuracy")
plt.show()

# Najbolji model

best_model_name = max(results, key=results.get)
print("\nNajbolji model:", best_model_name)